In [1]:
#Assignment 9: Gabriel Maldonado

In [2]:
import pygame
import moderngl
import numpy
import math
import glm
import assimp_py
from loadModelUsingAssimp_V2 import create3DAssimpObject

pygame.init()

width = 840
height = 480
pygame.display.gl_set_attribute(pygame.GL_CONTEXT_PROFILE_MASK, pygame.GL_CONTEXT_PROFILE_CORE)
pygame.display.gl_set_attribute(pygame.GL_MULTISAMPLESAMPLES, 16)
display = pygame.display.set_mode((width, height), flags= pygame.OPENGL | pygame.DOUBLEBUF | pygame.RESIZABLE)
display = pygame.display.set_caption(title = "Assignment 9 - Gabriel Maldonado")
gl = moderngl.get_context()
gl.info["GL_VERSION"]
#print(gl.max_samples) #16

C:\Users\vanes\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


'4.6.0 - Build 31.0.101.5186'

In [3]:
texture_vertex_shader_code = '''
    #version 330 core

    layout(location=0) in vec3 positions;
    layout(location=1) in vec3 in_normal;
    layout(location=2) in vec2 inUV;
    out vec2 f_uv;
    out vec3 f_normal;
    out vec3 f_position;
    uniform mat4 model;
    uniform mat4 viewMatrix;
    uniform mat4 perspectiveMatrix;

    void main(){
        vec4 position = model * vec4(positions, 1);
        gl_Position = perspectiveMatrix * viewMatrix * position;
        f_uv = inUV;
        f_position = position.xyz;
        mat3 normalMatrix = mat3(transpose(inverse(model)));
        f_normal = normalize(normalMatrix * in_normal);
}

'''

In [4]:
texture_fragment_shader_code = '''
    #version 330 core

    in vec3 f_normal;
    in vec3 f_position;
    in vec2 f_uv;
    out vec4 tColor;
    uniform vec3 eyePoint;
    uniform sampler2D map;
    uniform vec3 reflectance;
    uniform float shininess;
    uniform vec3 specularColor;
    uniform vec4 light;

    vec3 computeColor(){
        vec3 L;
        if (light.w > 0){
            L = normalize(light.xyz - f_position);
        }
        else{
            L = normalize(light.xyz);
        }
        vec3 N = normalize(f_normal);
        vec3 materialColor = texture(map, f_uv).rgb;
        vec3 color = vec3(0);
        if(dot(N,L) > 0) {

            vec3 diffuseColor = materialColor * dot(N,L);
            vec3 V = normalize(eyePoint - f_position);
            vec3 H = normalize(L + V);
            vec3 specularlyReflectedColor = specularColor * pow(dot(H,N), shininess);
            color = reflectance * diffuseColor + specularlyReflectedColor;
        }
        vec3 ambientColor = vec3(0.1) * materialColor * reflectance;
        color += ambientColor;
        return color;
}

    void main(){
        vec3 color = computeColor();
        tColor = vec4(color, 1);
}    
'''

In [5]:
program = gl.program(
    vertex_shader = texture_vertex_shader_code,
    fragment_shader = texture_fragment_shader_code
)

assimpObject = create3DAssimpObject("chair_table_class/scene.gltf", verbose = False, textureFlag = True, normalFlag = True)
material = assimpObject.scene.materials[0]
print(type(material))
print(material)


Accumulates Normals in the geomDataList
Accumulates Textures in the geomDataList
<class 'dict'>
{'NAME': 'Material.001', 'COLOR_DIFFUSE': [1.0, 1.0, 1.0, 1.0], 'NONE': 0.5, 'TEXTURE_BASE': 'textures/Material.001_baseColor.png', 'UVWSRC_BASE': 0, 'MAPPINGMODE_U_BASE': 0.0, 'MAPPINGMODE_V_BASE': 0.0, 'SHININESS': 20.0000057220459, 'COLOR_EMISSIVE': [0.0, 0.0, 0.0, 1.0], 'TWOSIDED': None, 'OPACITY': 1.0, 'SHADING_MODEL': 1.5414283107572988e-44, 'TEXTURES': {1: ['textures/Material.001_baseColor.png'], 12: ['textures/Material.001_baseColor.png']}}


In [6]:
format = "3f 3f 2f"
variables = ["positions", "in_normal", "inUV"]

renderables = assimpObject.getRenderables(gl, program, format, variables)
samplers = assimpObject.getSamplers(gl)
bounds = assimpObject.bound
gl.enable(gl.DEPTH_TEST)
pause = False
running = True
clock = pygame.time.Clock()
camera_elevation_angle = 30
camera_y_angle = 0
camera_angle_in_radians  = math.radians(90 - camera_elevation_angle)
distance = 2 * bounds.radius
displacement_vector = glm.vec3(0, distance * math.cos(camera_angle_in_radians), distance * math.sin(camera_angle_in_radians))
target_point = glm.vec3(bounds.center)
up_vector = glm.vec3(0,1,0)

lightComponent = 0
light_distance = 3 * bounds.radius
light_angle = 45
light_displacement_vector = glm.vec3(0, light_distance * math.cos(math.radians(45)), light_distance * math.sin(math.radians(45)))
light_vector = glm.rotate(light_displacement_vector, glm.radians(light_angle), glm.vec3(0,1,0))



#assimpObject._printSceneInfo()
object_reflectance = tuple(material.get("COLOR_DIFFUSE")[:3])
object_shininess = material.get("SHININESS", None)
if (object_shininess == None):
    object_shininess = 32.0
object_specularColor = material.get("COLOR_SPECULAR", None)
if (object_specularColor == None):
    object_specularColor = [1.0, 1.0, 1.0]
object_specularColor = tuple(object_specularColor[:3])

In [7]:


while running:
    for event in pygame.event.get():
        if (event.type == pygame.QUIT):
            running = False
        elif (event.type == pygame.KEYDOWN):
            if event.key == 27:
                running = False
            elif (event.key == pygame.K_p):
                pause = not pause
            elif (event.key == pygame.K_l):
                if (event.mod == pygame.KMOD_NONE):
                    lightComponent = 0
                    print("l lowercase")
                elif (event.mod == pygame.KMOD_LSHIFT):
                    lightComponent = 1
                    print("L uppercase")
                elif (event.mod == pygame.KMOD_RSHIFT):
                    lightComponent = 1
                    print("L uppercase")
            elif (event.key == pygame.K_LEFT):
                light_angle = light_angle - 5
                light_vector = glm.rotate(light_displacement_vector, glm.radians(light_angle), glm.vec3(0,1,0))
            elif (event.key == pygame.K_RIGHT):
                light_angle = light_angle + 5
                light_vector = glm.rotate(light_displacement_vector, glm.radians(light_angle), glm.vec3(0,1,0))
        elif (event.type == pygame.WINDOWRESIZED):
            width = event.x
            height = event.y

    aspect = width/height
    new_displacement_vector = glm.rotate(displacement_vector, glm.radians(camera_y_angle), glm.vec3(0,1,0))

    eye_point = target_point + new_displacement_vector
    viewMatrix = glm.lookAt(eye_point, target_point, up_vector)    

    camera_perspective_matrix = glm.perspective(math.radians(45), aspect, bounds.radius, 3 * bounds.radius)         
            
    gl.clear(0.5,0.5,0)

    model = glm.mat4(1)

    light = glm.vec4(light_vector, lightComponent)
    
    program["light"] = light
    program["viewMatrix"].write(viewMatrix)
    program["perspectiveMatrix"].write(camera_perspective_matrix)
    program["specularColor"] = object_specularColor
    program["eyePoint"] = eye_point
    program["reflectance"] = object_reflectance
    program["shininess"] = object_shininess
    assimpObject.render(program, renderables, samplers, M = model)

    
    pygame.display.flip()
    if not pause:
        clock.tick(30)
        camera_y_angle = camera_y_angle + 1
        if camera_y_angle > 360:
            camera_y_angle = 0
    
pygame.display.quit()


In [8]:
renderables

In [9]:
samplers